PREPARACIONES 


In [0]:
%sql
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

In [0]:
%sql
CREATE OR REPLACE TABLE tb_paises AS
SELECT * FROM VALUES
 ('PRT','Portugal'),('GBR','Reino Unido'),('FRA','Francia'),('ESP','España'),
 ('DEU','Alemania'),('ITA','Italia'),('IRL','Irlanda'),('BEL','Bélgica'),
 ('BRA','Brasil'),('NLD','Países Bajos'),('USA','Estados Unidos'),('CHE','Suiza'),
 ('AUT','Austria'),('SWE','Suecia'),('CHN','China'),('POL','Polonia'),
 ('RUS','Rusia'),('NOR','Noruega'),('DNK','Dinamarca'),('FIN','Finlandia'),
 ('ROU','Rumania'),('AUS','Australia'),('ARG','Argentina'),('LUX','Luxemburgo')
AS t(cod_pais, nombre_pais);

In [0]:
%sql
CREATE OR REPLACE TABLE tb_vendedores AS
SELECT * FROM VALUES
 (9,'Carlos Ruiz'),(240,'Ana Torres'),(1,'Luis Pérez'),(14,'María Gómez'),
 (7,'Jorge Salas'),(9999,'Vendedor sin reservas')
AS t(id_vendedor, nombre_vendedor);

In [0]:
%sql
CREATE OR REPLACE TABLE tb_reservas_id AS
SELECT ROW_NUMBER() OVER (ORDER BY hotel, arrival_date, lead_time, adr) AS id_reserva, *
FROM tb_reservas;

In [0]:
%sql
CREATE OR REPLACE TABLE tb_facturas AS
SELECT id_reserva,
       ROUND(adr * (stays_in_weekend_nights + stays_in_week_nights), 2) AS monto_facturado
FROM tb_reservas_id
WHERE id_reserva <= 2000
UNION ALL SELECT 9999991, CAST(150.0 AS DOUBLE)
UNION ALL SELECT 9999992, CAST(220.5 AS DOUBLE);

El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT COUNT(*) AS reservas_canceladas_ta_to,
       COUNT(*) * 1.5 AS perdida_usd
FROM tb_reservas
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;

Conclusión: Se pagaron USD 60228 a la agencia por 40152 reservas que luego se cancelaron. Ese dinero no genera ingresos, y justifica renegociar el contrato para pagar solo por reservas efectivas.

El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.


In [0]:
%sql
SELECT COUNT(*) AS cancelaciones_tardias,
       COUNT(*) * 120 AS perdida_total_usd
FROM tb_reservas
WHERE is_canceled = 1
  AND DATEDIFF(arrival_date, TRY_TO_DATE(reservation_status_date, 'd/M/yy')) < 3;

Conclusión: 3204 reservas se cancelaron con menos de 3 días de anticipación, con una pérdida operativa de USD 384480. Conviene endurecer las políticas de cancelación o pedir un depósito en reservas de último momento.

(JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql
SELECT r.is_canceled,
       p.nombre_pais
FROM tb_reservas r
LEFT JOIN tb_paises p
       ON r.country = p.cod_pais;

Conclusión: El LEFT JOIN conserva las 119390 reservas aunque el país no esté en el catálogo . Evita perder reservas por códigos de país faltantes.

(JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
%sql
SELECT p.nombre_pais,
       COUNT(*) AS reservas_canceladas
FROM tb_reservas r
INNER JOIN tb_paises p
        ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY reservas_canceladas DESC
LIMIT 5;

Conclusión: Portugal y Reino unido lideran las cancelaciones. Se pueden segmentar campañas para esos mercados.

(JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.


In [0]:
%sql
SELECT r.distribution_channel,
       v.nombre_vendedor
FROM tb_reservas r
FULL OUTER JOIN tb_vendedores v
             ON TRY_CAST(r.agent AS INT) = v.id_vendedor;

Conclusión: El FULL OUTER JOIN muestra reservas sin vendedor  y vendedores sin reservas. Permite detectar reservas sin responsable y vendedores inactivos.


(JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
%sql
SELECT f.id_reserva,
       f.monto_facturado,
       r.hotel
FROM tb_reservas_id r
RIGHT JOIN tb_facturas f
        ON r.id_reserva = f.id_reserva;

Conclusión: El RIGHT JOIN garantiza ver todas las facturas emitidas. Las que salen con hotel en NULL no tienen reserva asociada y deben revisarse.